# 🎨 DreamBooth LoRA Inference — ComfyUI + Custom Python

This notebook provides **two inference paths** for the fine-tuned DreamBooth LoRA model:

| Path | What it uses |
|------|--------------|
| **Part A — ComfyUI + ngrok** | Full ComfyUI web UI, publicly accessible via ngrok |
| **Part B — Custom Python** | Project's own `StableDiffusion` class, no extra UI |

> **Pre-requisite**: Training must be complete — `checkpoints/final_lora_weights.safetensors` must exist.
> The LoRA is auto-converted from PEFT training format → ComfyUI/A1111 format in Cell A3.

### Reference
Setup pattern based on [ComfyUI-Manager Colab notebook](https://github.com/ltdrdata/ComfyUI-Manager/blob/main/notebooks/comfyui_colab_with_manager.ipynb).

In [ ]:
# @title ⚙️ Environment Setup

USE_GOOGLE_DRIVE         = True   # @param {type:"boolean"}
UPDATE_COMFY_UI          = True   # @param {type:"boolean"}
USE_COMFYUI_MANAGER      = True   # @param {type:"boolean"}

# Path relative to MyDrive where the assignment lives
PROJECT_DRIVE_PATH = "training_zen/phase2/assignment2_4"  # @param {type:"string"}

import os, sys

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = f"/content/drive/MyDrive/{PROJECT_DRIVE_PATH}"
    COMFYUI_DIR  = "/content/drive/MyDrive/ComfyUI"
else:
    PROJECT_ROOT = "/content/assignment2_4"  # adjust if needed
    COMFYUI_DIR  = "/content/ComfyUI"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Project root : {PROJECT_ROOT}")
print(f"ComfyUI dir  : {COMFYUI_DIR}")

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU detected')


In [ ]:
# @title 🔑 Configuration
# Get your free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = ""    # @param {type:"string"}
COMFYUI_PORT     = 8188
BASE_MODEL_ID    = "stable-diffusion-v1-5/stable-diffusion-v1-5"

import yaml

LORA_WEIGHTS_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'final_lora_weights.safetensors')
TRAINING_CONFIG   = os.path.join(PROJECT_ROOT, 'dreamboth-lora-trainer', 'training_config.yaml')

with open(TRAINING_CONFIG) as f:
    train_cfg = yaml.safe_load(f)

LORA_RANK  = train_cfg['training'].get('lora_rank', 8)
LORA_ALPHA = float(train_cfg['training'].get('lora_alpha', 8))

if not os.path.exists(LORA_WEIGHTS_PATH):
    raise FileNotFoundError(
        f'LoRA weights not found: {LORA_WEIGHTS_PATH}\n'
        'Run training first (run_finetuning.ipynb).')

print(f'LoRA weights : {LORA_WEIGHTS_PATH}')
print(f'rank={LORA_RANK}, alpha={LORA_ALPHA}')
print(f'base model   : {BASE_MODEL_ID}')


---
## 🖥️ Part A — ComfyUI + ngrok

Cells **A1 → A4** install ComfyUI (with Manager), download the SD v1.5 base model,
convert your LoRA weights, and open a public URL via ngrok.

> Skip to **Part B** if you only need headless Python inference.

In [ ]:
# @title A1 — Install ComfyUI + ComfyUI-Manager
# Mirrors the official ComfyUI-Manager Colab setup pattern.

import os

# Clone or update ComfyUI
if not os.path.exists(COMFYUI_DIR):
    print('Cloning ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFYUI_DIR}
elif UPDATE_COMFY_UI:
    print('Updating ComfyUI...')
    !git -C {COMFYUI_DIR} pull --ff-only
else:
    print('ComfyUI present — skipping clone')

# ComfyUI-Manager (node manager UI)
if USE_COMFYUI_MANAGER:
    mgr = os.path.join(COMFYUI_DIR, 'custom_nodes', 'ComfyUI-Manager')
    if not os.path.exists(mgr):
        !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager {mgr}
    else:
        !git -C {mgr} pull --ff-only

# Python dependencies
!pip install -q accelerate einops 'transformers>=4.28.1' 'safetensors>=0.4.2' \
    aiohttp pyyaml Pillow scipy tqdm psutil 'tokenizers>=0.13.3' \
    kornia spandrel pyngrok soundfile sentencepiece

print('ComfyUI ready')


In [ ]:
# @title A2 — Download SD v1.5 Base Checkpoint (~4 GB)
# Stored on Google Drive so it survives Colab restarts.
# Skipped automatically when the file already exists.

from huggingface_hub import hf_hub_download
import os

ckpt_dir  = os.path.join(COMFYUI_DIR, 'models', 'checkpoints')
ckpt_name = 'v1-5-pruned-emaonly.safetensors'
ckpt_path = os.path.join(ckpt_dir, ckpt_name)
os.makedirs(ckpt_dir, exist_ok=True)

if os.path.exists(ckpt_path):
    size_gb = os.path.getsize(ckpt_path) / 1e9
    print(f'Checkpoint already on Drive ({size_gb:.1f} GB) — skipping download')
else:
    print(f'Downloading {ckpt_name} (~4 GB)...')
    hf_hub_download(
        repo_id   = BASE_MODEL_ID,
        filename  = ckpt_name,
        local_dir = ckpt_dir,
    )
    print(f'Saved to {ckpt_path}')


In [ ]:
# @title A3 — Convert LoRA to ComfyUI Format
#
# Training saves LoRA in PEFT format:
#   unet.base_model.model.<path>.lora_A.default.weight
#   unet.base_model.model.<path>.lora_B.default.weight
#
# ComfyUI / AUTOMATIC1111 expects:
#   lora_unet_<path_underscored>.lora_down.weight  (= lora_A)
#   lora_unet_<path_underscored>.lora_up.weight    (= lora_B)
#   lora_unet_<path_underscored>.alpha             (scalar)
#
# UNet target paths follow standard diffusers naming, e.g.:
#   down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q

import torch
from safetensors.torch import load_file, save_file

def convert_custom_lora_to_comfyui(src, dst, lora_alpha=8.0):
    weights = load_file(src)
    comfy   = {}
    alpha_t = torch.tensor(float(lora_alpha), dtype=torch.float32)
    n = 0
    for key, tensor in weights.items():
        if key.startswith('unet.base_model.model.'):
            inner, pfx = key[len('unet.base_model.model.'):], 'lora_unet'
        elif key.startswith('text_encoder.base_model.model.'):
            inner, pfx = key[len('text_encoder.base_model.model.'):], 'lora_te'
        else:
            continue
        if '.lora_A.default.weight' in inner:
            mod, suf = inner.replace('.lora_A.default.weight', ''), 'lora_down.weight'
        elif '.lora_B.default.weight' in inner:
            mod, suf = inner.replace('.lora_B.default.weight', ''), 'lora_up.weight'
        else:
            continue
        mk = mod.replace('.', '_')
        comfy[f'{pfx}_{mk}.{suf}'] = tensor
        ak = f'{pfx}_{mk}.alpha'
        if ak not in comfy:
            comfy[ak] = alpha_t.clone()
        n += 1
    save_file(comfy, dst)
    print(f'Converted {n} LoRA weight pairs ({len(comfy)} tensors total)')
    return comfy

lora_out_dir    = os.path.join(COMFYUI_DIR, 'models', 'loras')
os.makedirs(lora_out_dir, exist_ok=True)
COMFY_LORA_PATH = os.path.join(lora_out_dir, 'dreambooth_sks.safetensors')

convert_custom_lora_to_comfyui(LORA_WEIGHTS_PATH, COMFY_LORA_PATH, LORA_ALPHA)

print()
print('ComfyUI workflow (after opening the UI URL):')
print('  1. Right-click canvas > Add Node > loaders > Load LoRA')
print('  2. Wire: CheckpointLoaderSimple -> Load LoRA -> KSampler')
print('  3. lora_name : dreambooth_sks.safetensors')
print('  4. strength_model / strength_clip : 0.8')
print('  5. positive prompt: "a photo of sks person, ..."')


In [ ]:
# @title A4 — Launch ComfyUI + ngrok  ▶ Keep this cell running

import subprocess, threading, time, requests
from pyngrok import ngrok

if not NGROK_AUTH_TOKEN:
    raise ValueError('Set NGROK_AUTH_TOKEN in the Configuration cell!')

# Start ComfyUI
_proc = subprocess.Popen(
    ['python', 'main.py',
     '--port',   str(COMFYUI_PORT),
     '--listen', '0.0.0.0',
     '--disable-cuda-malloc',
     '--preview-method', 'auto'],
    cwd=COMFYUI_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def _logs(p):
    for line in p.stdout:
        print('[ComfyUI]', line, end='', flush=True)

threading.Thread(target=_logs, args=(_proc,), daemon=True).start()

# Wait for server to be ready
print('Waiting for ComfyUI', end='', flush=True)
for _ in range(90):
    try:
        if requests.get(f'http://127.0.0.1:{COMFYUI_PORT}/', timeout=1).ok:
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(2)
print(' ready!')

# Open ngrok tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
_tunnel = ngrok.connect(COMFYUI_PORT, 'http')

print()
print('=' * 55)
print('ComfyUI is live!')
print(f'Open this URL: {_tunnel.public_url}')
print('=' * 55)
print('Interrupt kernel (square button) to stop.')

try:
    _proc.wait()
except KeyboardInterrupt:
    _proc.terminate()
    ngrok.disconnect(_tunnel.public_url)
    ngrok.kill()
    print('ComfyUI and ngrok stopped.')


---
## 🐍 Part B — Custom Python Inference

Uses the project's own `StableDiffusion` class with PEFT LoRA — **no ComfyUI or web UI needed**.

**Run order:** B1 → B2 → B3 → (optional) B4

In [ ]:
# @title B1 — Imports & torchao shim

import os, sys, importlib.util

# ── Torchao compatibility shim ────────────────────────────────────────────────
# PEFT's LoRA dispatcher calls is_torchao_available() which raises ImportError
# when torchao is installed but below PEFT's minimum required version.
# Masking it tells PEFT the package is absent; it then uses the standard
# nn.Linear LoRA dispatcher instead.
_orig_fs = importlib.util.find_spec
def _mask_torchao(name, package=None):
    if name == 'torchao' or name.startswith('torchao.'): return None
    return _orig_fs(name, package)
importlib.util.find_spec = _mask_torchao

# ── sys.path: project root + trainer src ─────────────────────────────────────
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'dreamboth-lora-trainer')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── inject_lora via importlib (avoids hyphenated-dir import error) ────────────
_spec = importlib.util.spec_from_file_location(
    'model_utils',
    os.path.join(PROJECT_ROOT, 'dreamboth-lora-trainer', 'src', 'model_utils.py'))
_mu = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mu)
inject_lora = _mu.inject_lora

from models.stable_diffusion import StableDiffusion
print('Imports OK')


In [ ]:
# @title B2 — Load Pretrained SD v1.5 Weights + Inject & Load LoRA

import torch, gc
from safetensors.torch import load_file

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
QUANT  = 'fp16'  if DEVICE == 'cuda' else 'fp32'
print(f'Device: {DEVICE} ({QUANT})')

# 1. Instantiate and load HuggingFace pretrained weights into custom architecture
#    Prints: [CLIP] All keys loaded ✅  [VAE] ✅  [UNet] ✅
sd = StableDiffusion()
sd.load_weight(model_id=BASE_MODEL_ID, device=DEVICE, quantization=QUANT)

# 2. Inject LoRA adapter structure BEFORE loading trained weights
#    (creates lora_A / lora_B parameters so load_state_dict finds them)
sd.unet, sd.text_encoder = inject_lora(sd.unet, sd.text_encoder, train_cfg)
print('LoRA adapters injected')

# 3. Load trained LoRA weights from the checkpoint
lora_sd   = load_file(LORA_WEIGHTS_PATH, device=DEVICE)
unet_lora = {k[len('unet.'):]: v for k, v in lora_sd.items() if k.startswith('unet.')}
te_lora   = {k[len('text_encoder.'):]: v for k, v in lora_sd.items() if k.startswith('text_encoder.')}

res = sd.unet.load_state_dict(unet_lora, strict=False)
print(f'UNet LoRA : {len(unet_lora)} tensors | '
      f'missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}')

if te_lora:
    res = sd.text_encoder.load_state_dict(te_lora, strict=False)
    print(f'TE   LoRA : {len(te_lora)} tensors | '
          f'missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}')
else:
    print('Text encoder: no LoRA saved (train_text_encoder=false — expected)')

# 4. Switch to eval mode for inference
sd.unet.eval(); sd.text_encoder.eval(); sd.vae.eval()
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()
print('Model ready for inference')


In [ ]:
# @title B3 — Generate Image

import torch, numpy as np
from PIL import Image
from IPython.display import display

# ── Parameters (editable via Colab form) ─────────────────────────────────────
PROMPT          = 'a photo of sks person, highly detailed, sharp focus'  # @param {type:"string"}
NEGATIVE_PROMPT = 'blurry, low quality, deformed, ugly, watermark'       # @param {type:"string"}
NUM_STEPS       = 30   # @param {type:"slider", min:10, max:50, step:5}
GUIDANCE_SCALE  = 7.5  # @param {type:"slider", min:1.0, max:15.0, step:0.5}
SEED            = 42   # @param {type:"integer"}
IMAGE_SIZE      = 512  # must match training resolution

torch.manual_seed(SEED)
noise = torch.randn(1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8)

print(f'Generating  steps={NUM_STEPS}, CFG={GUIDANCE_SCALE}, seed={SEED}')
print(f'Prompt: {PROMPT}')

# Forward pass through the custom StableDiffusion model
# Runs: tokenize -> CLIP encode -> DDPM denoising loop -> VAE decode
with torch.inference_mode():
    raw = sd.forward(
        prompts          = [PROMPT],
        noise            = noise,
        num_steps        = NUM_STEPS,
        device           = DEVICE,
        guidance_scale   = GUIDANCE_SCALE,
        negative_prompts = [NEGATIVE_PROMPT],
    )  # shape (1, 3, H, W), range approx [-1, 1]

# Convert to uint8 PIL image
img_np  = ((raw.float().cpu().clamp(-1, 1) + 1) / 2
           ).squeeze(0).permute(1, 2, 0).numpy()  # (H, W, 3) in [0,1]
pil_img = Image.fromarray((img_np * 255).round().astype(np.uint8))

out_dir  = os.path.join(PROJECT_ROOT, 'inference_outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f'seed{SEED}_steps{NUM_STEPS}.png')
pil_img.save(out_path)
print(f'Saved: {out_path}')
display(pil_img)


In [ ]:
# @title B4 — (Optional) Batch Prompt Grid

import math, torch, numpy as np
from PIL import Image
from IPython.display import display

BATCH_PROMPTS = [
    'a photo of sks person smiling, portrait, studio lighting',
    'a photo of sks person in a park, natural lighting, candid',
    'a photo of sks person wearing a suit, professional headshot',
    'an oil painting of sks person, impressionist style, colorful',
]
BATCH_NEG   = 'blurry, low quality, deformed, watermark'
BATCH_STEPS = 25
BATCH_CFG   = 7.5

imgs = []
for i, prompt in enumerate(BATCH_PROMPTS):
    torch.manual_seed(SEED + i)
    n = torch.randn(1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8)
    print(f'[{i+1}/{len(BATCH_PROMPTS)}] {prompt[:65]}')
    with torch.inference_mode():
        r = sd.forward([prompt], n, BATCH_STEPS, DEVICE, BATCH_CFG, [BATCH_NEG])
    arr = ((r.float().cpu().clamp(-1,1)+1)/2).squeeze(0).permute(1,2,0).numpy()
    imgs.append(Image.fromarray((arr*255).round().astype(np.uint8)))

cols = min(2, len(imgs))
rows = math.ceil(len(imgs) / cols)
W, H = imgs[0].size
grid = Image.new('RGB', (cols*W, rows*H), (20, 20, 20))
for idx, img in enumerate(imgs):
    r2, c2 = divmod(idx, cols)
    grid.paste(img, (c2*W, r2*H))

grid_path = os.path.join(PROJECT_ROOT, 'inference_outputs', 'batch_grid.png')
grid.save(grid_path)
print(f'Grid saved: {grid_path}')
display(grid)
